# Goal of this notebook

- Develop python code for ingesting data from Space Weather API (and NOAA later).
- Once we are confident with our code, then we can:
1. try dumping some data to raw data folder
2. move developed code to `src/data`
3. create the corresponding `entrypoint` to interact with `src/data`

# Imports

In [1]:
import pandas as pd
import requests
import os
from dotenv import load_dotenv
import yaml
from datetime import datetime, timedelta
import time

import progressbar
from tqdm import tqdm


C:\Users\Keith\AppData\Local\Temp\ipykernel_33504\1877162822.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


# Ensure current working directory is the root

In [2]:
os.chdir("..")

In [3]:
ls

 Volume in drive C is Windows-SSD
 Volume Serial Number is 42FE-2E01

 Directory of c:\Users\Keith\OneDrive\Post uni\Personal projects\space-weather-project

02/12/2025  17:41    <DIR>          .
01/12/2025  12:40    <DIR>          ..
02/12/2025  17:41                47 .gitignore
02/12/2025  17:29    <DIR>          config
02/12/2025  17:30    <DIR>          data
02/12/2025  15:44                 4 docker-compose.yml
02/12/2025  15:44                 4 Dockerfile
02/12/2025  15:30    <DIR>          entrypoint
02/12/2025  17:17    <DIR>          env
02/12/2025  15:30    <DIR>          models
02/12/2025  16:11    <DIR>          notebooks
03/12/2025  15:51               540 README.md
02/12/2025  16:21               366 requirements-dev.txt
02/12/2025  15:44                 4 requirements-prod.txt
02/12/2025  15:30    <DIR>          src
02/12/2025  15:30    <DIR>          tests
               6 File(s)            965 bytes
              10 Dir(s)  38.558.208.000 bytes free


# Loading the config and API key

In [4]:
def load_config(path="config/local.yaml"):

    """
    Loads the YAML config file "~/config/local.yaml" for local development,
    and the "env/.env" file to retrieve actual API keys to be attached to the 
    config
    """
    
    # 1. load .env file
    load_dotenv(dotenv_path="env/.env")

    # 2. load YAML config
    # very similar to JSON (key value pairs)
    with open(path, "r") as f:
        config = yaml.safe_load(f)

    # 3. attach the real API key to config["space_weather"]

    # 3.1 retrieve the environment variable name
    api_env_var = config["space_weather"]["api_key_env"]

    # 3.2 retrieve API key from the env variable name (create another key)
    config["space_weather"]["api_key"] = os.getenv(api_env_var)

    return config

In [29]:
config = load_config()
sw_config = config['space_weather']
noaa_config = config['noaa']

sw_config

{'base_url': 'https://sws-data.sws.bom.gov.au/api/v1',
 'api_key_env': 'SPACE_WEATHER_API_KEY',
 'date_fmt': '%Y-%m-%d %H:%M:%S',
 'ingestion': {'k_index': {'timeout_s': 60,
   'raw_base_dir': 'data/01-raw/space_weather/k_index',
   'chunk_days': 30,
   'sleep_seconds': 5}},
 'endpoints': {'k_index': 'get-k-index',
  'a_index': 'get-a-index',
  'dst_index': 'get-dst-index'},
 'defaults': {'location_k': 'Australian region',
  'location_a': 'Australian region',
  'location_dst': 'Australian region'},
 'allowed_values': {'k_locations': ['Australian region',
   'Alice Springs',
   'Canberra',
   'Cocos Island',
   'Narrabri',
   'Darwin',
   'Hobart',
   'Launceston',
   'Learmonth',
   'Melbourne',
   'Norfolk Island',
   'Perth',
   'Sydney',
   'Townsville',
   'Casey',
   'Davis',
   'Macquarie Island',
   'Mawson']},
 'api_key': 'REMOVED_API_KEY'}

In [ ]:
1+2

In [7]:
sw_config.get('ingestion').get('k_index').get('raw_base_dir')

'data/01-raw/space_weather/k_index'

# Hit the k-index endpoint

Develop code first, then wrap it into a function


## Developing the code

Important notes from the Space Weather API doco

```
1. The location for which the K index data is required. Australian region, or an Australian region observing site: Alice Springs, Canberra, Cocos Island, Narrabri, Darwin, Hobart, Launceston, Learmonth, Melbourne, Norfolk Island, Perth, Sydney, Townsville, or an Antartic region observing site: Casey, Davis, Macquarie Island, Mawson.

2. Large historical data responses are limited to 10,000 elements; responses are truncated, with following items just omitted.
```

`LOCATION, START, END` should ideally be command line arguments in `entrypoint` (NOT config because `entrypoint` is the interface for the user running the system and so any knobs must be configured in `entrypoint`)


Some considerations
1. For a certain request with >10,000 entries (to which only the first 10k rows will be returned), since we know data is collected **every 3 hours**, *can we collect 10k rows at a time for large requests?*   
- ⚠️ Kernel died when processing a request with >10,000 entries (e.g. `LOCATION = "Australian region", START = '2015-01-01 00:00:00', END = '2024-01-01 00:00:00'`)
- Break them into even more smaller requests and use `time.sleep` in between micro-requests?

In [ ]:
# # Define two datetime objects
# dt1 = datetime(2025, 12, 3, 10, 0, 0)  # December 3, 2025, 10:00:00 AM
# dt2 = datetime(2025, 12, 3, 15, 30, 0)  # December 3, 2025, 3:30:00 PM

# # Calculate the difference
# time_difference = dt2 - dt1

# # Get the total seconds and convert to hours
# hours_difference = time_difference.total_seconds() / 3600

In [4]:
def fetch_k_index(location="Australian region",
                  start='2015-01-01 00:00:00',
                  end='2015-01-02 02:00:00'):
    
    """

    Performs POST request(s) to fetches K-index data from SW API's get-k-index endpoint
    for a given location, start and end dates.

    Things to consider
    1. pagination: fetch smaller chunks instead of retrieving everything in one request (even if # rows ~= 10k)

    -> Maybe for each function call you wanna write each chunk separately (rather than 
    writing the entire fetched data from the request at once). But consequence is that
    we need to check for overlaps/duplicates in the next set of requests

    -> Alternatively: impose a strict rule on the maximum chunk size in the raw data folder

    - 
    """
    
    # how many entries do we expect (since K-index are taken every 3 hours)


    # specify URL for resource locator, headers, request body
    url = f"https://sws-data.sws.bom.gov.au/api/v1/{sw_config['endpoints']['k_index']}"

    headers = {'Content-Type': 'application/json; charset=UTF-8'}

    # as mentioned, it would NOT be wise to retrieve everything in one sitting
    # (not possible anyway for large requests due to the 10k limit, and kernel tends to die
    # even for ~10k)
    requestBody = {
    'api_key': sw_config['api_key'],
    'options': {'location': location,
                'start': start,
                'end': end}
    }

    # make the POST request
    response = requests.post(url, headers=headers, json=requestBody)

    print(f"Executing POST request to retrieve K-index at {location} from {start} to {end}..")

    # 4. view the request results
    if response.status_code == 200:
        responseBody = response.json()
        data = responseBody['data']
        print("✅ Data successfully retrieved.")
        print("--- Raw data ----")
        print(data)
        print("--- Tabular Form ----")
        display(pd.DataFrame(data))
    else:
        responseBody = response.json()
        print("❌ Fail to retrieve data.")
        errors = responseBody['errors']
        print(errors)

In [ ]:
fetch_k_index()

Executing POST request to retrieve K-index at Australian region from 2015-01-01 00:00:00 to 2015-01-02 02:00:00..
✅ Data successfully retrieved.
--- Raw data ----
[{'index': 1, 'valid_time': '2015-01-01 00:00:00', 'analysis_time': '2015-01-02 01:30:00'}, {'index': 1, 'valid_time': '2015-01-01 03:00:00', 'analysis_time': '2015-01-02 01:30:00'}, {'index': 1, 'valid_time': '2015-01-01 06:00:00', 'analysis_time': '2015-01-02 01:30:00'}, {'index': 1, 'valid_time': '2015-01-01 09:00:00', 'analysis_time': '2015-01-02 01:30:00'}, {'index': 2, 'valid_time': '2015-01-01 12:00:00', 'analysis_time': '2015-01-02 01:30:00'}, {'index': 1, 'valid_time': '2015-01-01 15:00:00', 'analysis_time': '2015-01-02 01:30:00'}, {'index': 1, 'valid_time': '2015-01-01 18:00:00', 'analysis_time': '2015-01-02 01:30:00'}, {'index': 2, 'valid_time': '2015-01-01 21:00:00', 'analysis_time': '2015-01-02 01:30:00'}, {'index': 2, 'valid_time': '2015-01-02 00:00:00', 'analysis_time': '2015-01-03 01:30:00'}]
--- Tabular Form 

,index,valid_time,analysis_time
0,1,2015-01-01 00:00:00,2015-01-02 01:30:00
1,1,2015-01-01 03:00:00,2015-01-02 01:30:00
2,1,2015-01-01 06:00:00,2015-01-02 01:30:00
3,1,2015-01-01 09:00:00,2015-01-02 01:30:00
4,2,2015-01-01 12:00:00,2015-01-02 01:30:00
5,1,2015-01-01 15:00:00,2015-01-02 01:30:00
6,1,2015-01-01 18:00:00,2015-01-02 01:30:00
7,2,2015-01-01 21:00:00,2015-01-02 01:30:00
8,2,2015-01-02 00:00:00,2015-01-03 01:30:00


As discussed, we don't need to sort out duplicates right now, but we need a consistent naming system

## Same code, chunked by time & sleep
- Instead of breaking from the loop (and writing any previous requested data), recommended to **fail fast**: quickly raise an exception
- From SW API:
    - One of them is `None`
        - if `start` is `None`: fetch up to latest available data
        - if `end` is `None`: fetch from earliest available data
    - Both are `None`: only retrieve latest available data

**CONCLUSION: ONLY chunk if both `start` and `end` are non-empty**

- Use `_post_k_index` for a dedicated function for making a POST request (otherwise one needs to repeat twice for >= 1 `None` start/end)

In [ ]:
from datetime import datetime, timedelta
import time
import pandas as pd
import requests


# probably move to src/utils
def _fmt_dt(x):

    """
    Formats/validates string x to be compatible with SW API's 
    UTC date format "YYYY-MM-DD HH:mm:ss" 
    """

    if x is None:
        return None
    if isinstance(x, datetime):
        return x.strftime("%Y-%m-%d %H:%M:%S")
    if isinstance(x, str): # ideally we wanna check using regex
        return x  # assume already valid format
    raise TypeError(f"start/end must be str|datetime|None, got {type(x)}")


def _post_k_index(sw_config, location, start=None, end=None):

    """
    makes a POST request to the get-k-index API 

    location: string
    start: None/string or datetime of the form "YYYY-MM-DD HH:mm:ss"
    end: None or string of the form "YYYY-MM-DD HH:mm:ss"
    """
    # validate dates to have the correct format 
    start = _fmt_dt(start)
    end = _fmt_dt(end)

    # form the url from local config
    url = f"{sw_config['base_url']}/{sw_config['endpoints']['k_index']}"

    # form the request body
    headers = {"Content-Type": "application/json; charset=UTF-8"}

    options = {"location": location}
    if start is not None:
        options["start"] = start
    if end is not None:
        options["end"] = end

    body = {"api_key": sw_config["api_key"], "options": options}

    # retrieve the data 
    resp = requests.post(url, headers=headers, json=body)

    # fail fast if request fails in any way
    if resp.status_code != 200:
        raise RuntimeError(
            f"K-index request failed | location={location} start={start} end={end} "
            f"| status={resp.status_code} body={resp.text}"
        )


    return resp.json().get("data", [])


def fetch_k_index(sw_config, location, start=None, end=None):

    """
    Fetch K-index for [start, end) in chunks to avoid 10k limits.
    If at least one start/end is None, only a single request is made; 
    due to how SW API handles these edge cases.  
    
    start, end: strings of the form "YYYY-MM-DD HH:mm:ss"
    sw_config: a python Dictionary of local configs
    location: string of location to retrieve K-index from

    Returns an array of dicts.
    """

    # parse dates
    # if isinstance(start, str):
    #     start = datetime.fromisoformat(start)
    # if isinstance(end, str):
    #     end = datetime.fromisoformat(end)

    # quickly validate dates first
    start = _fmt_dt(start)
    end = _fmt_dt(end)

    # if start or end is missing (including both missing), do a single request
    # demorgans law: not ( (start is not None) and (end is not None) ) equiv to
    # (start is None) or (end is None)
    if (start is None) or (end is None):
        return _post_k_index(sw_config, location, start, end)

    # all start & end well defined: chunk days and sleep time from local config
    chunk_days = sw_config["ingestion"]["k_index"]["chunk_days"]
    sleep_s = sw_config["ingestion"]["k_index"]["sleep_seconds"]

    # quickly assert config inputs -- fail fast
    if chunk_days <= 0:
        raise ValueError("chunk_days must be > 0")
    if sleep_s < 0:
        raise ValueError("sleep_seconds must be >= 0")


    data = []

    # type cast to datetime first so that date shiftings are well defined
    current = datetime.fromisoformat(start)
    end = datetime.fromisoformat(end)

    # perform the request in chunk_days at a time, with some sleep time in between requests
    while current < end:

        chunk_end = min(current + timedelta(days=chunk_days), end)

               
        data_chunk = _post_k_index(sw_config, location, start=current, end=chunk_end)
        
        if data:
            data.extend(data_chunk)
        else:
            print("⚠️ Empty chunk")

    
        current = chunk_end

        # sleep, if specfied, to prevent overloading the API
        if sleep_s > 0:
            time.sleep(sleep_s)

    # if we have absolutely no data for the given request
    if not data:
        return []
    return data



## Next Corrected Version
- Created helper `_fmt_dt` validates input to ensure UTC date strings
- `_post_k_index` uses `_fmt_dt` to validate start and end dates
- `fetch_k_index` doesn't use `_fmt_dt`, only validates date arithmetic (uses `datetime.fromisoformat`)
- Note that if `start` and `end` are well defined (therefore request is chunked) the repeated `_post_k_index` calls will format the datetimes.
    - This is no problem as `_post_k_index` will not only be a mere helper but a function that can be called on its own (e.g. useful for testing)

Refactored function names
- `fetch_k_index` to `fetch_k_index_history` (more meaningful)
- additional `fetch_k_index_latest` to fetch ONLY the latest K-index, which by definition, is `_post_k_index(sw_config, location, start=None, end=None)`


Consider these changes:
- Progress bar rather than displaying raw dates

In [ ]:
def _fmt_dt(x):

    """
    Validates the argument to align to UTC date format "YYYY-MM-DD HH:mm:ss".
    """

    if x is None:
        return None
    
    if isinstance(x, datetime):
        return x.strftime("%Y-%m-%d %H:%M:%S")
    
    if isinstance(x, str):
        # validate that it successfully parses. 
        # O/w let fromisoformat() raise an Exception to fail fast
        datetime.fromisoformat(x)
        return x
    
    raise TypeError(...)


def _post_k_index(sw_config, location, start=None, end=None):

    """
    makes a POST request to hit the get-k-index API 

    location: string
    start: None/string or datetime of the form "YYYY-MM-DD HH:mm:ss"
    end: None or string of the form "YYYY-MM-DD HH:mm:ss"
    """
    # validate dates to have the correct format 
    start = _fmt_dt(start)
    end = _fmt_dt(end)

    # form the url from local config
    url = f"{sw_config['base_url']}/{sw_config['endpoints']['k_index']}"

    # form the request body
    headers = {"Content-Type": "application/json; charset=UTF-8"}

    options = {"location": location}
    if start is not None:
        options["start"] = start
    if end is not None:
        options["end"] = end

    body = {"api_key": sw_config["api_key"], "options": options}

    # retrieve the data 
    resp = requests.post(url, headers=headers, json=body)

    # fail fast if request fails in any way
    if resp.status_code != 200:
        raise RuntimeError(
            f"K-index request failed | location={location} start={start} end={end} "
            f"| status={resp.status_code} body={resp.text}"
        )


    return resp.json().get("data", [])



def fetch_k_index_history(sw_config, location, start=None, end=None):

    """
    Fetches K-index for a given location (historical or latest).
    If not None, start and end must have "YYYY-MM-DD HH:mm:ss" format.

    Behavior
    - start and end are not None & valid dates: K-index will be requested chunk_days at a time, with sleep_seconds sleep time in between
    - either start & end is None: SW API will consider earliest avaialble (start is None) or latest available (end is None)
    or latest available data (both start and end are None)
    """
    

    # Single-request cases (at least one start or end is None)
    if start is None or end is None:
        print(f"One of start and end dates are None. Performing a single POST request")
        return _post_k_index(sw_config, location, start, end)

    # From here on: start/end MUST be datetimes
    # to enable datetime arithmetic. 
    if isinstance(start, str):
        start = datetime.fromisoformat(start)
    if isinstance(end, str):
        end = datetime.fromisoformat(end)

    # read chunking parameters (duration per chunk & sleep time)
    chunk_days = sw_config["ingestion"]["k_index"]["chunk_days"]
    sleep_s = sw_config["ingestion"]["k_index"]["sleep_seconds"]

    if chunk_days <= 0:
        raise ValueError("chunk_days must be > 0")

    print(f"Retrieving K-index in {location},\nfrom {start} to {end} in chunks of {chunk_days} days,\nwith {sleep_s}s sleep time in between\n")

    data = []
    current = start

    # process the request in chunks
    bar = progressbar.ProgressBar(max_value=progressbar.UnknownLength)
    i = 0

    # current + k*chunk_days <= end; k += 1
    # at some point there will be k such that current + k*chunk_days >= end, which will be truncated to end
    # floor( (end - start) / chunk_days )
    while current < end:
        
        chunk_end = min(current + timedelta(days=chunk_days), end)

        print(f"Reterieving K-index from {current} to {chunk_end}")

        chunk = _post_k_index(
            sw_config,
            location,
            start=current,
            end=chunk_end
        )

        if chunk: # NOT append(). Equivalent to concatenating data = data + chunk
            data.extend(chunk)

        print(f"{len(data)} observations obtained so far.")

        current = chunk_end

        if sleep_s > 0:
            print(f"Sleeping for {sleep_s} secs..")
            time.sleep(sleep_s)


        # update progress bar
        bar.update(i); i += 1
        print('\n')

    print(f"K-index ingestion from {start} to {end} completed. {len(data)} observations obtained.")

    return data


def fetch_k_index_latest(sw_config, location="Australian region"):
    
    """
    Fetch the latest K-index for a given valid location.
    """
    print(f'Fetching the lates K-index at {location}')
    return _post_k_index(sw_config, location, start=None, end=None)







In [18]:
fetch_k_index_latest(sw_config)

[{'index': 3,
  'valid_time': '2025-12-21 09:00:00',
  'analysis_time': '2025-12-21 10:29:21'}]

In [38]:
fetch_k_index_history(sw_config, location="Australian region", start="2025-10-20", end="2025-12-01")

Retrieving K-index in Australian region,
from 2025-10-20 00:00:00 to 2025-12-01 00:00:00 in chunks of 30 days,
with 5s sleep time in between

Reterieving K-index from 2025-10-20 00:00:00 to 2025-11-19 00:00:00
240 observations obtained so far.
Sleeping for 5 secs..


/ |#                                                  | 0 Elapsed Time: 0:00:00




Reterieving K-index from 2025-11-19 00:00:00 to 2025-12-01 00:00:00
336 observations obtained so far.
Sleeping for 5 secs..


- |                                       #           | 1 Elapsed Time: 0:00:06




K-index ingestion from 2025-10-20 00:00:00 to 2025-12-01 00:00:00 completed. 336 observations obtained.


[{'index': 2,
  'valid_time': '2025-10-20 00:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 3,
  'valid_time': '2025-10-20 03:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 2,
  'valid_time': '2025-10-20 06:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 1,
  'valid_time': '2025-10-20 09:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 2,
  'valid_time': '2025-10-20 12:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 1,
  'valid_time': '2025-10-20 15:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 1,
  'valid_time': '2025-10-20 18:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 2,
  'valid_time': '2025-10-20 21:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 2,
  'valid_time': '2025-10-21 00:00:00',
  'analysis_time': '2025-10-22 04:02:27'},
 {'index': 2,
  'valid_time': '2025-10-21 03:00:00',
  'analysis_time': '2025-10-22 04:02:27'},
 {'index': 1,
  'valid_time': '2025-10-2

## Make more informative progress bar

Fixes
- Don't be a smartass to precompute # of chunks. Iterate first to obtain chunk endpoints
- Fixed `_fmt_dt` to ensure date is of the format `"YYYY-MM-DD HH:mm:ss"`
- Added timeout for POST request

Further consdierations
- Since K-index is inputted for every 3 hours, what if we want to request single-day historical data??

### Code

In [ ]:
def _fmt_dt(x):

    """
    Validates the argument to align to UTC date format "YYYY-MM-DD HH:mm:ss",
    which is what the SW API expects. But if argument is None then one simply
    returns None
    """

    if x is None:
        return None
    
    if isinstance(x, datetime):
        return x.strftime("%Y-%m-%d %H:%M:%S")
    
    if isinstance(x, str):
        dt = datetime.fromisoformat(x)
        return dt.strftime("%Y-%m-%d %H:%M:%S")

    
    raise TypeError(...)


def _post_k_index(sw_config, location, start=None, end=None):

    """
    makes a POST request to hit the get-k-index API 

    location: string
    start: None/string or datetime of the form "YYYY-MM-DD HH:mm:ss"
    end: None or string of the form "YYYY-MM-DD HH:mm:ss"
    """
    # validate dates to have the correct format 
    start = _fmt_dt(start)
    end = _fmt_dt(end)

    # form the url from local config
    url = f"{sw_config['base_url']}/{sw_config['endpoints']['k_index']}"

    # form the request body
    headers = {"Content-Type": "application/json; charset=UTF-8"}

    options = {"location": location}
    if start is not None:
        options["start"] = start
    if end is not None:
        options["end"] = end

    body = {"api_key": sw_config["api_key"], "options": options}

    # POST request to retrieve the data 
    # fail fast if request fails in any way
    try:
        resp = requests.post(url, headers=headers, json=body, timeout=60)
    except requests.RequestException as e:
        raise RuntimeError(
            f"K-index request failed (network) | location={location} start={start} end={end}"
    ) from e

    
    if resp.status_code != 200:
        raise RuntimeError(
            f"K-index request failed | location={location} start={start} end={end} "
            f"| status={resp.status_code} body={resp.text}"
        )


    return resp.json().get("data", [])



def fetch_k_index_history(sw_config, location, start=None, end=None):

    """
    
    Fetches K-index for a given location (historical or latest).
    If not None, start and end must have "YYYY-MM-DD HH:mm:ss" format.

    Behaviors
    - start and end are not None & valid dates: K-index will be requested chunk_days at a time, with sleep_seconds sleep time in between
    - either start & end is None: SW API will consider earliest avaialble (start is None) or latest available (end is None)
    or latest available data (both start and end are None)
    - 

    
    """
    

    # Single-request cases (at least one start or end is None)
    if start is None or end is None:
        print(f"One of start and end dates are None. Performing a single POST request")
        return _post_k_index(sw_config, location, start, end)

    # From here on: start/end MUST be datetimes
    # to enable datetime arithmetic. 
    if isinstance(start, str):
        start = datetime.fromisoformat(start)
    if isinstance(end, str):
        end = datetime.fromisoformat(end)

    # read chunking parameters (duration per chunk & sleep time)
    chunk_days = sw_config["ingestion"]["k_index"]["chunk_days"]
    sleep_s = sw_config["ingestion"]["k_index"]["sleep_seconds"]

    if chunk_days <= 0:
        raise ValueError("chunk_days must be > 0")

    print(f"Retrieving K-index in {location},\nfrom {start} to {end} in chunks of {chunk_days} days,\nwith {sleep_s}s sleep time in between\n")


    # rather than precomputing # chunks in advance (prone to errors unless
    # you are a god-tier infallible mathematician), iterate to store endpoints
    # of chunks in advance.

    current = start
    chunks = [] # storing the temporal endpoints for each chunk

    while current < end:
        chunk_end = min(current + timedelta(days=chunk_days), end)
        chunks.append((current, chunk_end))
        current = chunk_end

    data = [] # storing the actual K-index data

    for current, chunk_end in tqdm(chunks):

        #print(f"Reterieving K-index from {current} to {chunk_end}")

        chunk = _post_k_index(
            sw_config,
            location,
            start=current,
            end=chunk_end
        )

        if chunk:
            # NOT append().
            # Equivalent to concatenating chunks = chunks + chunk
            data.extend(chunk)

        print(f"{len(data)} observations obtained so far.")

        if sleep_s > 0:
            print(f"Sleeping for {sleep_s} secs..\n")
            time.sleep(sleep_s)


    print(f"K-index ingestion from {start} to {end} completed. {len(data)} observations obtained.")

    return data


def fetch_k_index_latest(sw_config, location="Australian region"):
    
    """
    Fetch the latest K-index for a given valid location.
    """
    print(f'Fetching the lates K-index at {location}')
    return _post_k_index(sw_config, location, start=None, end=None)







### Testing for a few arguments

In [52]:
fetch_k_index_history(sw_config, location="Australian region", start="2025-10-20", end="2025-12-01")

Retrieving K-index in Australian region,
from 2025-10-20 00:00:00 to 2025-12-01 00:00:00 in chunks of 30 days,
with 5s sleep time in between



  0%|          | 0/2 [00:00<?, ?it/s]

240 observations obtained so far.
Sleeping for 5 secs..



 50%|█████     | 1/2 [00:06<00:06,  6.92s/it]

336 observations obtained so far.
Sleeping for 5 secs..



100%|██████████| 2/2 [00:13<00:00,  6.91s/it]

K-index ingestion from 2025-10-20 00:00:00 to 2025-12-01 00:00:00 completed. 336 observations obtained.


[{'index': 2,
  'valid_time': '2025-10-20 00:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 3,
  'valid_time': '2025-10-20 03:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 2,
  'valid_time': '2025-10-20 06:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 1,
  'valid_time': '2025-10-20 09:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 2,
  'valid_time': '2025-10-20 12:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 1,
  'valid_time': '2025-10-20 15:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 1,
  'valid_time': '2025-10-20 18:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 2,
  'valid_time': '2025-10-20 21:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 2,
  'valid_time': '2025-10-21 00:00:00',
  'analysis_time': '2025-10-22 04:02:27'},
 {'index': 2,
  'valid_time': '2025-10-21 03:00:00',
  'analysis_time': '2025-10-22 04:02:27'},
 {'index': 1,
  'valid_time': '2025-10-2

In [59]:
fetch_k_index_history(sw_config, location="Australian region", start="2025-10-20", end=None)

One of start and end dates are None. Performing a single POST request


[{'index': 2,
  'valid_time': '2025-10-20 00:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 3,
  'valid_time': '2025-10-20 03:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 2,
  'valid_time': '2025-10-20 06:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 1,
  'valid_time': '2025-10-20 09:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 2,
  'valid_time': '2025-10-20 12:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 1,
  'valid_time': '2025-10-20 15:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 1,
  'valid_time': '2025-10-20 18:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 2,
  'valid_time': '2025-10-20 21:00:00',
  'analysis_time': '2025-10-21 00:18:20'},
 {'index': 2,
  'valid_time': '2025-10-21 00:00:00',
  'analysis_time': '2025-10-22 04:02:27'},
 {'index': 2,
  'valid_time': '2025-10-21 03:00:00',
  'analysis_time': '2025-10-22 04:02:27'},
 {'index': 1,
  'valid_time': '2025-10-2

In [60]:
fetch_k_index_history(sw_config, location="Australian region", start=None, end="2021-10-20")

One of start and end dates are None. Performing a single POST request


[{'index': 4,
  'valid_time': '1999-03-10 00:00:00',
  'analysis_time': '1999-03-11 01:30:00'},
 {'index': 4,
  'valid_time': '1999-03-10 03:00:00',
  'analysis_time': '1999-03-11 01:30:00'},
 {'index': 5,
  'valid_time': '1999-03-10 06:00:00',
  'analysis_time': '1999-03-11 01:30:00'},
 {'index': 5,
  'valid_time': '1999-03-10 09:00:00',
  'analysis_time': '1999-03-11 01:30:00'},
 {'index': 3,
  'valid_time': '1999-03-10 12:00:00',
  'analysis_time': '1999-03-11 01:30:00'},
 {'index': 3,
  'valid_time': '1999-03-10 15:00:00',
  'analysis_time': '1999-03-11 01:30:00'},
 {'index': 3,
  'valid_time': '1999-03-10 18:00:00',
  'analysis_time': '1999-03-11 01:30:00'},
 {'index': 3,
  'valid_time': '1999-03-10 21:00:00',
  'analysis_time': '1999-03-11 01:30:00'},
 {'index': 3,
  'valid_time': '1999-03-11 00:00:00',
  'analysis_time': '1999-03-12 01:30:00'},
 {'index': 3,
  'valid_time': '1999-03-11 03:00:00',
  'analysis_time': '1999-03-12 01:30:00'},
 {'index': 4,
  'valid_time': '1999-03-1

In [61]:
fetch_k_index_history(sw_config, location="Australian region", start=None, end=None)

One of start and end dates are None. Performing a single POST request


[{'index': 2,
  'valid_time': '2025-12-29 00:00:00',
  'analysis_time': '2025-12-29 01:20:15'}]

Test output for single day requests.

In [58]:
fetch_k_index_history(sw_config, location="Australian region", start="2025-12-02", end="2025-12-03")

Retrieving K-index in Australian region,
from 2025-12-02 00:00:00 to 2025-12-03 00:00:00 in chunks of 30 days,
with 5s sleep time in between



  0%|          | 0/1 [00:00<?, ?it/s]

8 observations obtained so far.
Sleeping for 5 secs..



100%|██████████| 1/1 [00:07<00:00,  7.52s/it]

K-index ingestion from 2025-12-02 00:00:00 to 2025-12-03 00:00:00 completed. 8 observations obtained.


[{'index': 2,
  'valid_time': '2025-12-02 00:00:00',
  'analysis_time': '2025-12-03 00:17:15'},
 {'index': 2,
  'valid_time': '2025-12-02 03:00:00',
  'analysis_time': '2025-12-03 00:17:15'},
 {'index': 2,
  'valid_time': '2025-12-02 06:00:00',
  'analysis_time': '2025-12-03 00:17:15'},
 {'index': 1,
  'valid_time': '2025-12-02 09:00:00',
  'analysis_time': '2025-12-03 00:17:15'},
 {'index': 1,
  'valid_time': '2025-12-02 12:00:00',
  'analysis_time': '2025-12-03 00:17:15'},
 {'index': 2,
  'valid_time': '2025-12-02 15:00:00',
  'analysis_time': '2025-12-03 00:17:15'},
 {'index': 2,
  'valid_time': '2025-12-02 18:00:00',
  'analysis_time': '2025-12-03 00:17:15'},
 {'index': 2,
  'valid_time': '2025-12-02 21:00:00',
  'analysis_time': '2025-12-03 00:17:15'}]

## Incorporate Saving logic

- Consider the directory to save to as an argument, 

In [ ]:
# def write_manifest(sw_config, location, start, end, run_dir):

#     # e.g.
#     # run_dir = "data/01-raw/space_weather/k_index/run_id=...__loc=...__start=...__end=..."

#     # expected manifest content (some can be read from the config, but we possibly
#     # need to add some to the config)
#         # - `source`: `"space_weather"`
#         # - `dataset`: `"k_index"`
#         # - `location`
#         # - `start`, `end` (as strings)
#         # - `chunk_days`,
#         # - `sleep_seconds`
#         # - `base_url`,
#         # - `endpoint`
#         # - `run_id` (timestamp/uuid)

#     # success indication: via another _SUCCESS file (no double writes)

#     with open(os.path.join(run_dir, "_manifest.json"), 'w') as f:
#         # write the file
#         pass


# def ingest_k_index_run(sw_config, location, start, end, run_dir):

#     write_manifest(sw_config, location, start, end, run_dir)

#     # the idea: 
#     # -> iter_k_index_chunks fetches (from the SW API) and yields data
#     # -> write_chunk only writes EXISTING FETCHED data.
    
#     for chunk_start, chunk_end, chunk_data in iter_k_index_chunks(...):
#         write_chunk(run_dir, chunk_start, chunk_end, chunk_data)


#     write_success(run_dir)


## Moving to `src`

### Imports

In [59]:
from __future__ import annotations

import json
import os
import time
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, Iterator, List, Optional, Tuple

import requests

try:
    from tqdm import tqdm
except ImportError:  # keep src light; tqdm is optional
    def tqdm(x):  # type: ignore
        return x

### Time formatting

In [82]:
def _fmt_dt_for_api(sw_config: Dict[str, Any], x: Optional[object]) -> Optional[str]:
    """
    Returns a string in a format compatible to SW API, listed
    in the config `sw_config`.
    E.g. "YYYY-MM-DD HH:mm:ss" (UTC) if x is str/datetime.
    
    Returns None if x is None.
    Raises TypeError/ValueError for unsupported formats.
    """
    if x is None:
        return None

    _SW_API_DT_FMT = sw_config['date_fmt']

    if isinstance(x, datetime):
        # assume already UTC-ish; if tz-aware, convert to UTC
        if x.tzinfo is not None:
            x = x.astimezone(timezone.utc).replace(tzinfo=None)
        return x.strftime(_SW_API_DT_FMT)

    if isinstance(x, str):
        # validate + normalize
        dt = datetime.fromisoformat(x)  # may raise ValueError -> fail fast
        if dt.tzinfo is not None:
            dt = dt.astimezone(timezone.utc).replace(tzinfo=None)
        return dt.strftime(_SW_API_DT_FMT)

    raise TypeError(f"start/end must be str|datetime|None, got {type(x)}")


def _parse_dt(x: object) -> datetime:
    """
    Parse str/datetime into a naive datetime (assumed UTC) for arithmetic.
    """
    if isinstance(x, datetime):
        if x.tzinfo is not None:
            x = x.astimezone(timezone.utc).replace(tzinfo=None)
        return x
    if isinstance(x, str):
        dt = datetime.fromisoformat(x)
        if dt.tzinfo is not None:
            dt = dt.astimezone(timezone.utc).replace(tzinfo=None)
        return dt
    raise TypeError(f"Expected str|datetime, got {type(x)}")


def _run_id_utc() -> str:
    """
    Returns the current datetime in UTC format with non alphanumerics removed.

    Main use case is an identifier proxy (generating run ids)
    """
    # Example: 20251229T103210Z
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")


def _chunk_token(dt_: Optional[datetime]) -> str:

    """
    Returns the provided in UTC format with non alphanumerics removed.
    
    Main use case is for chunk filenames
    """

    if dt_ is None:
        return "open"
    # Example: 20250101T000000Z
    return dt_.replace(tzinfo=timezone.utc).strftime("%Y%m%dT%H%M%SZ")

### Core HTTP POST call

In [92]:
# -----------------------------
# Core HTTP call (single POST)
# -----------------------------

def post_k_index(
    sw_config: Dict[str, Any],
    location: str,
    start: Optional[object] = None,
    end: Optional[object] = None,
    #timeout_s: int = 60,
) -> List[Dict[str, Any]]:
    """
    Single POST to get-k-index. start/end may be None/str/datetime.
    Returns: list[dict] (the 'data' array).
    """

    # 1. make the request body

    start_s = _fmt_dt_for_api(sw_config, start)
    end_s = _fmt_dt_for_api(sw_config, end)

    url = f"{sw_config['base_url'].rstrip('/')}/{sw_config['endpoints']['k_index']}"
    headers = {"Content-Type": "application/json; charset=UTF-8"}

    options: Dict[str, Any] = {"location": location}
    if start_s is not None:
        options["start"] = start_s
    if end_s is not None:
        options["end"] = end_s

    body = {"api_key": sw_config["api_key"], "options": options}

    try:
        #print(f"Performing POST request to get-k-index..")
        timeout_s = sw_config.get('ingestion').get('k_index').get('timeout_s', 60)
        resp = requests.post(url, headers=headers, json=body, timeout=timeout_s)

    except requests.RequestException as e:

        raise RuntimeError(
            f"K-index request failed (network) | location={location} start={start_s} end={end_s}"
        ) from e

    if resp.status_code != 200:
        
        raise RuntimeError(
            f"K-index request failed | location={location} start={start_s} end={end_s} "
            f"| status={resp.status_code} body={resp.text}"
        )
    
    #print(f"POST request success.")
    return resp.json().get("data", [])

### Chunk iterator

In [99]:
# -----------------------------
# Chunk iterator: fetch + yield
# -----------------------------

@dataclass(frozen=True)
class KIndexChunk:
    chunk_start: Optional[datetime]
    chunk_end: Optional[datetime]
    data: List[Dict[str, Any]]


def iter_k_index_chunks(
    sw_config: Dict[str, Any],
    location: str,
    start: Optional[object] = None,
    end: Optional[object] = None,
) -> Iterator[KIndexChunk]:
    
    """
    Yields KIndexChunk(s). This (generator) function performs the POST requests.

    Rules:
    - If start is None OR end is None -> exactly ONE request (no chunking).
      The SW API infers missing endpoints.
    - If both provided -> chunk across [start, end) using config chunk_days/sleep_seconds.

    """
    # 1. Single-request path
    if start is None or end is None:
        
        # 1.1 perform the POST request 
        print(f"\nOne of start or end date is None. Performing a single POST request")
        data = post_k_index(sw_config, location, start=start, end=end)

        # 1.2 return a custom KIndexChunk
        chunk_start_dt = _parse_dt(start) if start is not None else None
        chunk_end_dt = _parse_dt(end) if end is not None else None

        print(f"✔️ (Single-request) data fetched successfully!")
        yield KIndexChunk(chunk_start_dt, chunk_end_dt, data)
        return

    # 2. Chunking path for other cases
    start_dt = _parse_dt(start)
    end_dt = _parse_dt(end)

    # 2.1 quickly handle invalid cases
    if start_dt > end_dt:
        raise ValueError(f"start must be <= end. Got start={start_dt}, end={end_dt}")

    chunk_days = sw_config["ingestion"]["k_index"]["chunk_days"]
    sleep_s = sw_config["ingestion"]["k_index"]["sleep_seconds"]

    if not isinstance(chunk_days, int) or chunk_days <= 0:
        raise ValueError("ingestion.k_index.chunk_days must be a positive int")
    if not isinstance(sleep_s, (int, float)) or sleep_s < 0:
        raise ValueError("ingestion.k_index.sleep_seconds must be >= 0")

    current = start_dt

    # 2.2 Handle start == end (single “point-in-time” request)
    # yield one zero-length chunk (often returns empty)
    if start_dt == end_dt:
        print(f"\nFetching K-index at a point in time on {start_dt.strftime('%B %d, %Y, %I:%M:%S %p')}")
        data = post_k_index(sw_config, location, start=start_dt, end=end_dt)
        print(f"✔️ (Single-request) data fetched successfully!")
        yield KIndexChunk(start_dt, end_dt, data)
        return
    
    # 2.3 start < end, fetch the data by chunking
    while current < end_dt:
        chunk_end = min(current + timedelta(days=chunk_days), end_dt)

        print(f"\nFetching K-index chunk from {current.strftime('%B %d, %Y, %I:%M:%S %p')} to {chunk_end.strftime('%B %d, %Y, %I:%M:%S %p')}..")

        data = post_k_index(sw_config, location, start=current, end=chunk_end)

        yield KIndexChunk(current, chunk_end, data)

        current = chunk_end

        if sleep_s > 0:
            print(f"✔️ Chunk fetched successfully! Sleeping for {sleep_s}s..\n")
            time.sleep(float(sleep_s))



### Disk writes

In [93]:
# -----------------------------
# Disk writes: manifest/chunks/success
# -----------------------------

def _atomic_write_json(path: Path, payload: Dict[str, Any]) -> None:

    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
    tmp.replace(path)


def write_manifest(
    run_dir: Path,
    *,
    sw_config: Dict[str, Any],
    location: str,
    start: Optional[object],
    end: Optional[object],
    run_id: str,
    status: str,
    extra: Optional[Dict[str, Any]] = None,
) -> None:
    
    """
    Writes/updates _manifest.json atomically.
    """
    run_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = run_dir / "_manifest.json"

    payload: Dict[str, Any] = {
        "source": "space_weather",
        "dataset": "k_index",
        "run_id": run_id,
        "created_at_utc": run_id,  # run_id is already a UTC timestamp string
        "status": status,          # RUNNING | SUCCESS | FAILED
        "location": location,
        "start": _fmt_dt_for_api(sw_config, start),
        "end": _fmt_dt_for_api(sw_config, end),
        "chunk_days": sw_config.get("ingestion", {}).get("k_index", {}).get("chunk_days"),
        "sleep_seconds": sw_config.get("ingestion", {}).get("k_index", {}).get("sleep_seconds"),
        "base_url": sw_config.get("base_url"),
        "endpoint": sw_config.get("endpoints", {}).get("k_index"),
    }

    print(f"JSON manifest to be saved:\n{json.dumps(payload, indent=2)}")

    if extra:
        payload.update(extra)

    #print(f"Writing {manifest_path} to disk..")

    _atomic_write_json(manifest_path, payload)

    #print(f"Write success.")


def chunk_filename(chunk_start: Optional[datetime], chunk_end: Optional[datetime]) -> str:
    """
    Naming convention:
      - both None => chunk_latest.jsonl
      - missing one side => open token
      - else => chunk_<start>__<end>.jsonl
    """
    if chunk_start is None and chunk_end is None:
        return "chunk_latest.jsonl"
    return f"chunk_{_chunk_token(chunk_start)}__{_chunk_token(chunk_end)}.jsonl"


def write_chunk_jsonl(
    run_dir: Path,
    *,
    chunk_start: Optional[datetime],
    chunk_end: Optional[datetime],
    chunk_data: List[Dict[str, Any]],
) -> Path:
    """
    Writes a chunk as JSONL (one JSON object per line).
    """
    run_dir.mkdir(parents=True, exist_ok=True)
    fname = chunk_filename(chunk_start, chunk_end)
    path = run_dir / fname

    # Atomic-ish write for chunk files too
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        for row in chunk_data:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    tmp.replace(path)

    return path


def write_success(run_dir: Path) -> None:
    (run_dir / "_SUCCESS").write_text("", encoding="utf-8")


def write_failed(run_dir: Path, message: str) -> None:
    (run_dir / "_FAILED").write_text(message, encoding="utf-8")

### Main ingestion run

In [94]:
# -----------------------------
# Main ingestion "run"
# -----------------------------

def ingest_k_index_run(
    sw_config: Dict[str, Any],
    *,
    location: str,
    start: Optional[object] = None,
    end: Optional[object] = None,
    raw_base_dir: Optional[object] = None,
) -> Path:
    """
    End-to-end ingestion run:
      - creates run_dir under raw_base_dir/run_id=...
      - writes manifest (RUNNING)
      - iterates chunks (fetches) and writes chunk files
      - writes _SUCCESS + manifest (SUCCESS)
      - if exception: writes _FAILED + manifest (FAILED), then re-raises

    Notes:
    - As per SW API, datetimes, if provided, must be at UTC format.

    Example call:
    run_dir = ingest_k_index_run(
        sw_config,
        location="Australian region",
        start="2021-12-01 00:00:00",
        end="2022-01-01 00:00:00",
    )
    print("Wrote run to:", run_dir)
    """

    # 1. write metadata
    run_id = _run_id_utc()

    if raw_base_dir is None:
        # You can place this in YAML instead; keeping default helps early dev.
        #raw_base_dir = sw_config.get("raw_base_dir", "data/01-raw/space_weather/k_index")
        raw_base_dir = sw_config.get('ingestion').get('k_index').get('raw_base_dir')

    base = Path(raw_base_dir)
    run_dir = base / f"run_id={run_id}"

    print(f"Writing initial metadata..")

    write_manifest(
        run_dir,
        sw_config=sw_config,
        location=location,
        start=start,
        end=end,
        run_id=run_id,
        status="RUNNING",
    )

    print(f"✔️ Initial metadata written.\n")

    try:
        # 2.a.1 fetch chunks and store & write one at a time 
        total_rows = 0
        chunk_files: List[str] = []

        # will stream a List of KIndexChunks w attributes e.g. chunk_start, chunk_end, data
        chunks_iter = iter_k_index_chunks(sw_config, location, start=start, end=end)

        # Do NOT list(chunks_iter) unless we want to store all fetched data in memory.
        # when this loop is executed,
        # iter_k_index_chunks() execute up to 1st yield -> 1st loop iteration
        # -> iter_k_index_chunks() resume up to second yield -> 2nd loop iteration
        # ...
        for chunk in tqdm(chunks_iter,
                          desc="K-index chunks processed",
                          unit=" chunk"):            
            
            print(f'Writing this chunk to disk..')
            out_path = write_chunk_jsonl(
                run_dir,
                chunk_start=chunk.chunk_start,
                chunk_end=chunk.chunk_end,
                chunk_data=chunk.data,
            )
            print(f'Write succeeded.')

            chunk_files.append(out_path.name)
            
            total_rows += len(chunk.data)

            print(f'# observations so far: {total_rows}.')

        # 2.a.2 confirm success of ingestion

        write_success(run_dir)

        # 2.a.3 update the manifest/metadata
        write_manifest(
            run_dir,
            sw_config=sw_config,
            location=location,
            start=start,
            end=end,
            run_id=run_id,
            status="SUCCESS",
            extra={"total_rows": total_rows, "chunk_files": chunk_files},
        )

        print(f'✅ Run succeeded (saved at {run_dir})')

        return run_dir

    except Exception as e:
        
        # 2.b.1 if any Exception occurs, fail fast 
        write_failed(run_dir, repr(e))

        # 2.b.2
        write_manifest(
            run_dir,
            sw_config=sw_config,
            location=location,
            start=start,
            end=end,
            run_id=run_id,
            status="FAILED",
            extra={"error": repr(e)},
        )
        raise

## Initial run

### Success scenarios

#### Valid locations, start, end

In [106]:
print(2 if 10 < 3 else None)

None


In [100]:
run_dir = ingest_k_index_run(
    sw_config,
    location="Australian region",
    start="2021-11-01 00:00:00",
    end="2022-01-01 00:00:00",
)

Writing initial metadata..
JSON manifest to be saved:
{
  "source": "space_weather",
  "dataset": "k_index",
  "run_id": "20260107T135618Z",
  "created_at_utc": "20260107T135618Z",
  "status": "RUNNING",
  "location": "Australian region",
  "start": "2021-11-01 00:00:00",
  "end": "2022-01-01 00:00:00",
  "chunk_days": 30,
  "sleep_seconds": 5,
  "base_url": "https://sws-data.sws.bom.gov.au/api/v1",
  "endpoint": "get-k-index"
}
✔️ Initial metadata written.



K-index chunks processed: 0 chunk [00:00, ? chunk/s]


Fetching K-index chunk from November 01, 2021, 12:00:00 AM to December 01, 2021, 12:00:00 AM..


K-index chunks processed: 1 chunk [00:02,  2.59s/ chunk]

Writing this chunk to disk..
Write succeeded.
# observations so far: 240.
✔️ Chunk fetched successfully! Sleeping for 5s..


Fetching K-index chunk from December 01, 2021, 12:00:00 AM to December 31, 2021, 12:00:00 AM..


...

Writing this chunk to disk..
Write succeeded.
# observations so far: 480.
✔️ Chunk fetched successfully! Sleeping for 5s..


Fetching K-index chunk from December 31, 2021, 12:00:00 AM to January 01, 2022, 12:00:00 AM..


K-index chunks processed: 3 chunk [00:14,  5.31s/ chunk]

Writing this chunk to disk..
Write succeeded.
# observations so far: 488.
✔️ Chunk fetched successfully! Sleeping for 5s..



K-index chunks processed: 3 chunk [00:19,  6.60s/ chunk]

JSON manifest to be saved:
{
  "source": "space_weather",
  "dataset": "k_index",
  "run_id": "20260107T135618Z",
  "created_at_utc": "20260107T135618Z",
  "status": "SUCCESS",
  "location": "Australian region",
  "start": "2021-11-01 00:00:00",
  "end": "2022-01-01 00:00:00",
  "chunk_days": 30,
  "sleep_seconds": 5,
  "base_url": "https://sws-data.sws.bom.gov.au/api/v1",
  "endpoint": "get-k-index"
}
✅ Run succeeded (saved at data\01-raw\space_weather\k_index\run_id=20260107T135618Z)


#### One of `start` or `end` is `None`

In [101]:
run_dir = ingest_k_index_run(
    sw_config,
    location="Australian region",
    start="2024-11-01 00:00:00",
    end=None,
)

Writing initial metadata..
JSON manifest to be saved:
{
  "source": "space_weather",
  "dataset": "k_index",
  "run_id": "20260107T135652Z",
  "created_at_utc": "20260107T135652Z",
  "status": "RUNNING",
  "location": "Australian region",
  "start": "2024-11-01 00:00:00",
  "end": null,
  "chunk_days": 30,
  "sleep_seconds": 5,
  "base_url": "https://sws-data.sws.bom.gov.au/api/v1",
  "endpoint": "get-k-index"
}
✔️ Initial metadata written.



K-index chunks processed: 0 chunk [00:00, ? chunk/s]


One of start or end date is None. Performing a single POST request


K-index chunks processed: 1 chunk [00:02,  2.95s/ chunk]

✔️ (Single-request) data fetched successfully!
Writing this chunk to disk..
Write succeeded.
# observations so far: 3461.
JSON manifest to be saved:
{
  "source": "space_weather",
  "dataset": "k_index",
  "run_id": "20260107T135652Z",
  "created_at_utc": "20260107T135652Z",
  "status": "SUCCESS",
  "location": "Australian region",
  "start": "2024-11-01 00:00:00",
  "end": null,
  "chunk_days": 30,
  "sleep_seconds": 5,
  "base_url": "https://sws-data.sws.bom.gov.au/api/v1",
  "endpoint": "get-k-index"
}
✅ Run succeeded (saved at data\01-raw\space_weather\k_index\run_id=20260107T135652Z)


In [102]:
run_dir = ingest_k_index_run(
    sw_config,
    location="Australian region",
    start=None,
    end="2022-01-01 00:00:00",
)

Writing initial metadata..
JSON manifest to be saved:
{
  "source": "space_weather",
  "dataset": "k_index",
  "run_id": "20260107T135720Z",
  "created_at_utc": "20260107T135720Z",
  "status": "RUNNING",
  "location": "Australian region",
  "start": null,
  "end": "2022-01-01 00:00:00",
  "chunk_days": 30,
  "sleep_seconds": 5,
  "base_url": "https://sws-data.sws.bom.gov.au/api/v1",
  "endpoint": "get-k-index"
}
✔️ Initial metadata written.



K-index chunks processed: 0 chunk [00:00, ? chunk/s]


One of start or end date is None. Performing a single POST request


K-index chunks processed: 1 chunk [00:02,  2.70s/ chunk]

✔️ (Single-request) data fetched successfully!
Writing this chunk to disk..
Write succeeded.
# observations so far: 10000.
JSON manifest to be saved:
{
  "source": "space_weather",
  "dataset": "k_index",
  "run_id": "20260107T135720Z",
  "created_at_utc": "20260107T135720Z",
  "status": "SUCCESS",
  "location": "Australian region",
  "start": null,
  "end": "2022-01-01 00:00:00",
  "chunk_days": 30,
  "sleep_seconds": 5,
  "base_url": "https://sws-data.sws.bom.gov.au/api/v1",
  "endpoint": "get-k-index"
}
✅ Run succeeded (saved at data\01-raw\space_weather\k_index\run_id=20260107T135720Z)


## Failure scenarios

#### Invalid location, valid start, valid end

In [70]:
run_dir = ingest_k_index_run(
    sw_config,
    location="Australian regionn",
    start=None,
    end="2022-01-01 00:00:00",
)

Writing initial metadata..
JSON manifest to be saved:
{
  "source": "space_weather",
  "dataset": "k_index",
  "run_id": "20260107T092737Z",
  "created_at_utc": "20260107T092737Z",
  "status": "RUNNING",
  "location": "Australian regionn",
  "start": null,
  "end": "2022-01-01 00:00:00",
  "chunk_days": 30,
  "sleep_seconds": 5,
  "base_url": "https://sws-data.sws.bom.gov.au/api/v1",
  "endpoint": "get-k-index"
}
Initial metadata written.



0it [00:00, ?it/s]

One of start or end date is None. Performing a single POST request


0it [00:00, ?it/s]

JSON manifest to be saved:
{
  "source": "space_weather",
  "dataset": "k_index",
  "run_id": "20260107T092737Z",
  "created_at_utc": "20260107T092737Z",
  "status": "FAILED",
  "location": "Australian regionn",
  "start": null,
  "end": "2022-01-01 00:00:00",
  "chunk_days": 30,
  "sleep_seconds": 5,
  "base_url": "https://sws-data.sws.bom.gov.au/api/v1",
  "endpoint": "get-k-index"
}


RuntimeError: K-index request failed | location=Australian regionn start=None end=2022-01-01 00:00:00 | status=400 body={"errors":[{"code":43,"message":"Unsupported option value: location=Australian regionn"}]}

#### Valid location, start > end

In [71]:
run_dir = ingest_k_index_run(
    sw_config,
    location="Australian region",
    start="2022-01-01 00:00:00",
    end="2021-01-01 00:00:00",
)

Writing initial metadata..
JSON manifest to be saved:
{
  "source": "space_weather",
  "dataset": "k_index",
  "run_id": "20260107T092957Z",
  "created_at_utc": "20260107T092957Z",
  "status": "RUNNING",
  "location": "Australian region",
  "start": "2022-01-01 00:00:00",
  "end": "2021-01-01 00:00:00",
  "chunk_days": 30,
  "sleep_seconds": 5,
  "base_url": "https://sws-data.sws.bom.gov.au/api/v1",
  "endpoint": "get-k-index"
}
Initial metadata written.



0it [00:00, ?it/s]

JSON manifest to be saved:
{
  "source": "space_weather",
  "dataset": "k_index",
  "run_id": "20260107T092957Z",
  "created_at_utc": "20260107T092957Z",
  "status": "FAILED",
  "location": "Australian region",
  "start": "2022-01-01 00:00:00",
  "end": "2021-01-01 00:00:00",
  "chunk_days": 30,
  "sleep_seconds": 5,
  "base_url": "https://sws-data.sws.bom.gov.au/api/v1",
  "endpoint": "get-k-index"
}


ValueError: start must be <= end. Got start=2022-01-01 00:00:00, end=2021-01-01 00:00:00

In [103]:
bool(not None)

True